# Figure panels for tracks data processing and effective $k_\mathrm{obs}$ fits

## Imports

- Calcium tracks data processing functions are in the `analysis/tracks.jl` module. 
- Functions that make final, detailed plots are in `analysis/tracksPlots.jl`. 
- Functions to fit a one-step model for effective activation rates $k_\mathrm{obs}$ are in `analysis/modelsML.jl`. 

In [ ]:
using Plots: Plots as pl
using Printf: @sprintf
using Measures: mm

# Data processing
module tracksAnalysis
    include("analysis/tracks.jl")
end
using .tracksAnalysis: preptrajdata, findactivtimes_thresh, 
    cleannames, get_csv_file_names,  prepalldsets, concfromstr,
    prepare_all_cdfs

# import small utilities from the plotting scripts
module tracksPlots
    include("analysis/tracksPlots.jl")
end
using .tracksPlots: plot_trajs_activ_cdf


# Model definitions and log-likelihood
module modelsML
    include("analysis/modelsML.jl")
end
using .modelsML: expocdf, expopdf, expoccdf, logexpoccdf, logexpopdf, 
    expo_loglhood_1dset, hess_minuslogl_expo!, deriv_minuslogl_expo!


In [ ]:
# Default styles
pl.default(
    labelfontsize = 14,
    tickfontsize = 12,
    framestyle = :axes,  # https://docs.juliaplots.org/stable/gallery/gr/generated/gr-ref034/
    yminorgrid = false,
    xminorgrid = false,
    xgrid = false,
    ygrid = false,
    size = (550, 450),
    linewidth = 2.0,
    legendfontsize=10, 
    fontfamily="Arial"
)

# Custom line styles with shorter dashes
ddash = [8, 4]
dotdash = [4, 4, 4]

# Default analysis parameters
default_fluo_factor = 0.8
cdfthreshold = 0.75
# Filtering parameters: maximal start time, minimal traj length
minlength = 50
maxstarttime = 30.0
filter_params = [minlength, maxstarttime]

# Example trajectories and thresholding
All trajectories in light gray on one plot (all with annotated transition time perhaps), one colored with thresholding to emphasize the procedure. 

In [ ]:
# Main fitting and plotting function for one example
function plot_tracks_thresholding(
        filepath, plotfolder; 
        highlight=1, 
        fluothresh=default_fluo_factor, 
        filterpms=filter_params, 
        cdfthresh=cdfthreshold, 
        dosave=false
    )
    # highlight: which track to highlight. 
    greentrajs = preptrajdata(filepath; ch="MEAN_INTENSITY_CH2", filterpms=filterpms)  # Calcium fluorescence
    fi = basename(filepath)

    activtimes, activfluos, thresh = findactivtimes_thresh(
        greentrajs; fracabove=fluothresh)
    backgnd = thresh / (1.0 + fluothresh)

    # Make plot
    shortnames, lblconcs = cleannames([fi])
    fi2 = replace(shortnames[1], "." => "-")
    fi2 = replace(fi2, " " => "_")
    ppp = plot_trajs_activ_cdf(greentrajs, activtimes, activfluos, thresh, backgnd, 
                    highlight; conclbl=shortnames[1])
    
    display(ppp)
    if dosave
        pl.savefig(ppp, joinpath(plotfolder, "tracks_example_thresholding_$(fi2).pdf"))
        println("Saved tracks examples")
    end
end

In [ ]:
datadir = joinpath("data", "JR064_tracks")
plotfolder = joinpath("figures", "final_tracks_panels")

# Get a cleaned up, sorted list of data file names and paths
fnames, fpaths = get_csv_file_names(datadir)
shortnames, lblconcs = cleannames(fnames)

dset_id = 3
chosenfile = fpaths[dset_id]
println("File analyzed: ", fnames[dset_id])

plot_tracks_thresholding(
    chosenfile, plotfolder;
    highlight=12, 
    dosave=true, 
)

# Effective activation rate $k_\mathrm{obs}$ fitting


## Analysis functions


TODO: move the fitting functions to a module. Split modelsML.jl into two modules, one for k_obs, one for hit models, containing both the model definitions and the fitting scripts. 

In [ ]:
using Optim: Optim, optimize, IPNewton

In [ ]:
### RMS ERROR COMPUTATION ###

"""
Compute the root-mean-squared error of the fitted cumulative distribution 
function for each dataset TODO

Arguments:
- p_all: fitted parameters array, shape [2, n_datasets]
  and t0_n is the activation time offset for dataset n.
- tu_all: list of unique time arrays for each dataset
- cumuls_all: list of empirical cumulative distributions for each dataset
- ilim0s_all: list of lower indices (first point with t > 0) for each dataset
- ilim1s_all: list of upper indices (last point to include) for each dataset
- cdffct: the CDF function to use (e.g., expocdf)

Returns:
- rmserr: the root-mean-squared error computed over all datasets combined
"""
function compute_rms_all_dsets(p_all, tu_all, cumuls_all, ilim0s_all, 
                                ilim1s_all, cdffct)
    ndsets = length(tu_all)
    if ndsets !== size(p_all, 2)
        error("Number of datasets and parameters not matching")
    end
    rms_errors = zeros(Float64, ndsets)
    sres_total = 0.0
    npts_total = 0

    for n in 1:ndsets
        tu = tu_all[n]
        cumul = cumuls_all[n]
        ilim0 = ilim0s_all[n]
        ilim1 = ilim1s_all[n]
        k = p_all[1, n]
        t0 = p_all[2, n]

        # Determine starting index for RMS calculation
        rms_i = (ilim1 > ilim0) ? ilim0 : 1

        # Compute the fitted CDF for this dataset
        cdfestim = cdffct(tu[rms_i:ilim1], k; t0=t0)

        # Accumulate squared residuals
        dataset_sres = sum((cumul[rms_i:ilim1] .- cdfestim).^2)
        dataset_npts = (ilim1 - rms_i + 1)
        rms_errors[n] = sqrt(dataset_sres / dataset_npts)

        sres_total += dataset_sres
        npts_total += dataset_npts
    end

    # Compute overall RMS error for the full fit across all datasets
    overall_rms = sqrt(sres_total / npts_total)
    return rms_errors, overall_rms
end


In [ ]:
### FITTING FUNCTIONS ###

function fitkobs(tactiv_alldsets, dsetnames, durations, cdfthresh)
    n_dsets = length(tactiv_alldsets)
    prepres = prepare_all_cdfs(tactiv_alldsets, dsetnames, durations, cdfthresh)
    concs_all, ncell_all, k0_all, k1_all, te_all, tu_all, cumuls_all = prepres

    fit_params = zeros(Float64, 2, n_dsets)
    fit_params_errors = zeros(Float64, 2, n_dsets)
    for i in 1:n_dsets
        lower = [1e-6, -20.0]
        upper = [10.0, tu_all[i][1]-1e-3]
        init_p = [0.1, -0.1]

        klim0 = isnothing(k0_all[i]) ? 1 : k0_all[i]
        klim1 = k1_all[i]
        te = te_all[i]

        cost = p -> -expo_loglhood_1dset(p, 
                            tu_all[i], ncell_all[i], klim0, klim1; te=te)
        minusgrad! = (g, p) -> deriv_minuslogl_expo!(g, p, 
                                tu_all[i], ncell_all[i], klim0, klim1; te=te)
        minushess! = (h, p) -> hess_minuslogl_expo!(h, p, 
                                tu_all[i], ncell_all[i], klim0, klim1; te=te)
        inner_optimizer = IPNewton()
        results = optimize(cost, minusgrad!, minushess!, 
                            lower, upper, init_p, inner_optimizer)

        if !Optim.converged(results)
            println("Dataset $(dsetnames[i]) did not converge for cdfthresh = $cdfthresh")
        end
        params = Optim.minimizer(results)
        fit_params[:, i] = params

        # Get variance of these parameter fits. Hessian is not positive definite
        # because the likelihood is monotonically increasing with t0, 
        # so best fit is t0 = boundary, k* derived from dlogL/dk = 0
        # So the uncertainty on the slope is estimated by inverting the FIM = -hessian^-1 element, 
        # while that on t0 is the uncertainty on an experimental time
        # so let's say (0.5 * time bin)^2. 
        minushessmat = zeros(2, 2)
        minushess!(minushessmat, params)
        varislope = 1.0 / minushessmat[1, 1]
        varit0 = (0.5 * (tu_all[i][2] - tu_all[i][1]))^2
        fit_params_errors[:, i] = [sqrt(varislope), sqrt(varit0)]

    end

    rms_errors, overall_rms = compute_rms_all_dsets(fit_params, 
                                tu_all, cumuls_all, k0_all, k1_all, expocdf)
    return fit_params, fit_params_errors, rms_errors, overall_rms, prepres
end



## Plotting functions

TODO: move the relevant, final ones to tracksPlots.jl. Maybe split tracksPlots.jl into two modules, one for kobs, one for the hits models. 

In [ ]:
### PLOTTING FUNCTIONS ###

"""
Create a single-panel plot comparing the empirical normalized CDFs to
fitted exponential model curves for each dataset.

Arguments:
- cdf_prep_results: output from prepare_all_cdfs
- nicenames: list of dataset names
- fit_params: 2 x n_dsets array of [k; t0] per dataset
- cdffct: exponential CDF function to plot
- overall_rms: overall RMS error for the fit
- durations: list of experiment durations
"""
function plot_kobs_fits(cdf_prep_results, nicenames, 
            fit_params, cdffct, overall_rms, durations)
    (concs_all, ncell_all, k0_all, k1_all, 
                te_all, tu_all, cumuls_all) = cdf_prep_results
    n_dsets = length(tu_all)

    lblconcs = [concfromstr(name, "pM", ' ') for name in nicenames]
    if any(x -> x === nothing, lblconcs)
        error("Some dataset names do not contain a parsable concentration value")
    end
    conc_labels = [@sprintf("%.3g", c) for c in lblconcs]
    colors = pl.cgrad(:magma)

    tplotlim = [-5.0, maximum([maximum(a) for a in tu_all]) * 1.05]
    fracplotlim = [-0.05, 1.05]

    p = pl.plot(left_margin=3mm, right_margin=3mm,
                bottom_margin=3mm, xlabel="Time (s)", ylabel="Fraction of activated cells",
                xlim=tplotlim, ylim=fracplotlim, legend=:bottomright,
                tickfont=pl.font(13), legendfontsize=11,
                titlefontsize=12, legendmargin=4mm, foreground_color_legend=nothing,
                title=raw"1-step model for $k_\mathrm{obs}$ fits")

    # Plot all data points first
    markertypes = [:circle, :diamond, :square, :utriangle]
    for i in 1:n_dsets
        tu = tu_all[i]
        cumul = cumuls_all[i]

        goodt = vcat([0.0], tu)
        ncumul = vcat([0.0], cumul)

        k = fit_params[1, i]
        t0 = fit_params[2, i]
        trange = collect(LinRange(0, durations[i], 200))
        yfit = cdffct(trange, k; t0=t0)
        line_color = colors[(2*i-2)/(2*n_dsets-1)]
        conc_label = nicenames[i]

        pl.plot!(p, goodt, ncumul, linetype=:steppost, 
                 color=:grey, linewidth=3.0, alpha=0.8, label=nothing)
        pl.scatter!(p, tu, cumul, color=:grey, markersize=4, m=markertypes[i],
                    alpha=0.6, label="Data $conc_label")
        pl.plot!(p, trange, yfit, label=raw"Fit " * conc_label,
                 color=line_color, linewidth=4.4, linestyle=:solid)
    end

    return p
end


"""
Plot fitted k_obs for each dataset as a function of concentration. 
"""
function plot_kobs_vs_concentration(nicenames, fit_params, fit_errors, ncell_all)
    lblconcs = [concfromstr(name, "pM", ' ') for name in nicenames]
    valid = findall(x -> x !== nothing, lblconcs)
    if length(valid) != length(lblconcs)
        error("Some dataset names do not contain a parsable concentration value")
    end
    
    unique_concs = unique(sort(lblconcs))
    fit_ks_errs = fit_errors[1, :]
    fitted_ks = fit_params[1, :]
    selected_concs = lblconcs
    n_dsets = length(selected_concs)
    mark_colors = pl.cgrad(:magma)[[(2*i-2)/(2*n_dsets-1) for i in 1:1:n_dsets]]
    p = pl.plot(selected_concs, fitted_ks, linewidth=2, color=:grey, label=nothing)
    pl.scatter!(p, selected_concs, fitted_ks, yerr=fit_ks_errs, marker=:circle, markersize=7,
                color=mark_colors, xlabel="Particle concentration (pM)", 
                ylabel=raw"Effective rate $k_{\mathrm{obs}} (\mathrm{s^{-1}})$", 
                legend=false, left_margin=3mm, right_margin=3mm, bottom_margin=3mm)
    
    # Compare to diffusion-limited rate
    k0 = 0.16  # For B cell radius 2.5 um
    c_line = collect(LinRange(minimum(lblconcs)-0.005, minimum(lblconcs) + 0.2, 100))
    k_line = k0 .* c_line
    pl.plot!(p, c_line, k_line, linestyle=:dot, color=:black,
             linewidth=2.0)
    
    # Annotate "Diffusion-limited"
    ytxt, xtxt = minimum(fitted_ks)+0.001, minimum(lblconcs)+1.5
    pl.plot!(p, [xtxt, xtxt-1.0], [ytxt, ytxt], 
        arrow=true, color=:black, linewidth=2, label=nothing)
    pl.annotate!(p, xtxt+0.2, ytxt, ("Diffusion-limited", :left, 12))

    # Also annotate cell activation? No, this is a big assumption, just don't annotate
    #ytxt, xtxt = maximum(fitted_ks), maximum(lblconcs)-1.5
    #pl.annotate!(p, xtxt-0.2, ytxt, ("Cell activation-limited?", :right, 12))
    return p
end



## Main $k_\mathrm{obs}$ analysis function

In [ ]:

### MAIN ANALYSIS FUNCTION ###

function fit_ml_all_datasets(; 
    datadir="data/JR064_tracks",
    plotdir="figures/ml_fits_kobs_10pMnorm/",
    fracabove=default_fluo_factor,
    cdfthresh=cdfthreshold)
    
    # Create output directory if it doesn't exist
    if !isdir(plotdir)
        mkpath(plotdir)
    end

    fnames, fpaths = get_csv_file_names(datadir)

    println("\n" * "="^70)
    println("MAXIMUM LIKELIHOOD FITTING - JR064 DATASETS")
    println("="^70)
    println("\nLoading and preprocessing datasets...")
    
    # Step 1: Preprocess all datasets
    tactiv_alldsets, durations, nicenames = prepalldsets(fpaths; 
        fracabove=fracabove, disp=false)
    n_dsets = length(tactiv_alldsets)
    println("\nLoaded $n_dsets datasets:")
    for (i, name) in enumerate(nicenames)
        println("  [$i] $name")
    end

    # Step 2: Fit the exponential model to each dataset separately
    println("\n" * "-"^70)
    println("Fitting exponential model for each dataset separately...")
    println("-"^70)
    fitres = fitkobs(tactiv_alldsets, nicenames, durations, cdfthresh)
    fit_params, fit_errors, rms_errors, overall_rms, prep_cdf_res = fitres

    println("\nExponential fit results:")
    for i in 1:n_dsets
        println(@sprintf("  Dataset %d (%s): k = %.4f s^-1, t0 = %.4f s, RMS = %.3e", 
                         i, nicenames[i], fit_params[1,i], fit_params[2,i], rms_errors[i]))
    end
    println(@sprintf("\nOverall RMS = %.3e", overall_rms))

    # Select one dataset per concentration for plotting
    selected_indices = Int[7, 4, 3, 1]
    sind = join((string(a) for a in selected_indices), "")
    selected_tactiv = tactiv_alldsets[selected_indices]
    selected_durations = durations[selected_indices]
    selected_names = nicenames[selected_indices]
    selected_fit_params = fit_params[:, selected_indices]
    selected_fit_errors = fit_errors[:, selected_indices]

    println("\nSelected $(length(selected_indices)) datasets for plotting:")
    for (j, name) in enumerate(selected_names)
        println("  [$j] $name")
    end

    # Step 3: Create comparison plots
    println("\n" * "-"^70)
    println("Generating plots...")
    println("-"^70)

    selected_cdf_prep = collect(a[selected_indices] for a in prep_cdf_res)

    p_all = plot_kobs_fits(selected_cdf_prep, selected_names, selected_fit_params, 
                        expocdf, overall_rms, selected_durations)
    plot_file = joinpath(plotdir, "exp_cdf_all_datasets_$(sind).pdf")
    pl.savefig(p_all, plot_file)
    display(p_all)
    println("  Saved: $plot_file")

    p_conc = plot_kobs_vs_concentration(selected_names, 
                selected_fit_params, selected_fit_errors, prep_cdf_res[2])
    plot_file = joinpath(plotdir, "exp_k_vs_concentration_$(sind).pdf")
    pl.savefig(p_conc, plot_file)
    display(p_conc)
    println("  Saved: $plot_file")

    println("\n" * "="^70)
    println("Analysis complete!")
    println("="^70 * "\n")

    return (fit_params, overall_rms, nicenames, tactiv_alldsets)
end



In [ ]:
fit_ml_all_datasets();